<a href="https://colab.research.google.com/github/gohard-lab/gohard_ai_upscaler/blob/main/gohard_AI_upscaler.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# @title 1. 🛠️ Setup AI Upscaling Engine (Gohard_AI_Upscaler v2.1)
# Note for Global Users :This notebook is optimized for a global audience. While the video tutorial might show a Korean interface in some parts, the engine and code logic are fully English-compatible. Follow the instructions below to start your upscaling workflow.

import os, shutil
from IPython.display import clear_output

# Core dependencies setup
!pip install -q streamlit streamlit_javascript uv
!uv pip install basicsr facexlib gfpgan supabase Image pathlib

# Refresh Real-ESRGAN directory for a clean install
if os.path.exists('/content/Real-ESRGAN'):
    shutil.rmtree('/content/Real-ESRGAN')

!git clone https://github.com/xinntao/Real-ESRGAN.git /content/Real-ESRGAN

%cd /content/Real-ESRGAN

# Download internal usage tracker
!wget -q -O tracker_web_colab.py https://raw.githubusercontent.com/gohard-lab/ai_image_upscaler/refs/heads/main/src/tracker_web_colab.py

# 4. Install requirements and apply monkey patch for torchvision compatibility
!uv pip install -r requirements.txt
!python setup.py develop
!sed -i 's/from torchvision.transforms.functional_tensor import rgb_to_grayscale/from torchvision.transforms.functional import rgb_to_grayscale/g' /usr/local/lib/python3.*/dist-packages/basicsr/data/degradations.py

clear_output()
print("✅ Installation and environment setup complete!")

✅ Installation and environment setup complete!


In [2]:
# @title 2. ⚙️ Enhancement & Storage Path Configuration
# @markdown ### 📂 Storage & Workflow Settings
Use_Google_Drive = True # @param {type:"boolean"}
# @markdown > If enabled, upscaled results and CSV reports will be permanently saved to the `Gohard_Upscaler_Results` folder in your Google Drive.

Mode = "Pro Mode (Batch + Mirroring)" # @param ["Basic Mode (Single)", "Pro Mode (Batch + Mirroring)"]
Target_Canvas = "Instagram (1080x1080)" # @param ["None", "4K (3840x2160)", "Instagram (1080x1080)"]
Background_Color = "Black" # @param ["Black", "White", "Gray"]
Sort_Failures = True # @param {type:"boolean"}

# Logic Mappings
color_map = {"Black": (0,0,0), "White": (255,255,255), "Gray": (128,128,128)}
canvas_map = {"None": None, "4K (3840x2160)": (3840, 2160), "Instagram (1080x1080)": (1080, 1080)}

print(f"✅ Configuration Applied: Google Drive({Use_Google_Drive}) / Mode({Mode})")

✅ Configuration Applied: Google Drive(True) / Mode(Pro Mode (Batch + Mirroring))


In [3]:
# @title 3. 🚀 Execute AI Upscaling (Gohard_AI_Upscaler v2.2)
import os, shutil, time, csv, cv2, torch
import json
from google.colab import files, drive
from tracker_web_colab import log_app_usage
from pathlib import Path
from PIL import Image
from datetime import datetime

%cd /content/Real-ESRGAN

def start_upscale():
    # Initial logging for workflow analytics
    log_app_usage("upscaler_web", "process_started", details={"canvas": str(Target_Canvas)})

    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

    if Use_Google_Drive:
        drive.mount('/content/drive')
        base_output = Path('/content/drive/MyDrive/Gohard_Upscaler_Results') / timestamp
    else:
        base_output = Path('/content/Real-ESRGAN/results') / timestamp

    upload_dir = Path('/content/Real-ESRGAN/upload')
    log_file = base_output / f"report_{timestamp}.csv"

    # Refresh upload directory for the current session
    if upload_dir.exists(): shutil.rmtree(upload_dir)
    upload_dir.mkdir(parents=True, exist_ok=True)
    base_output.mkdir(parents=True, exist_ok=True)

    print(f"📂 Results for this session will be saved at: {base_output}")
    print("👇 Select the images you want to restore (Batch upload supported)")
    uploaded = files.upload()

    for filename in uploaded.keys():
        shutil.move(filename, upload_dir / filename)

    # Initialize CSV Report Header
    with open(log_file, 'w', newline='') as f:
        csv.writer(f).writerow(["Filename", "Canvas", "Time(s)", "Status"])

    images = list(upload_dir.rglob('*'))
    valid_exts = ['.jpg', '.jpeg', '.png']

    start_total = time.time()
    for img_path in images:
        if img_path.suffix.lower() not in valid_exts: continue
        img_start = time.time()

        status = "Unknown"
        error_msg = ""

        try:
            rel_path = img_path.relative_to(upload_dir)
            save_path = base_output / rel_path
            save_path.parent.mkdir(parents=True, exist_ok=True)

            # --- AI Engine Core (Real-ESRGAN + GFPGAN) ---
            !python inference_realesrgan.py -n RealESRGAN_x4plus -i "{img_path}" -o "{save_path.parent}" --outscale 4 --face_enhance --tile 400

            # --- Canvas Post-processing ---
            res_img_path = save_path.parent / f"{img_path.stem}_out{img_path.suffix}"
            if canvas_map[Target_Canvas]:
                with Image.open(res_img_path) as up_img:
                    canvas = Image.new("RGB", canvas_map[Target_Canvas], color_map[Background_Color])
                    up_img.thumbnail(canvas_map[Target_Canvas], Image.LANCZOS)
                    offset = ((canvas_map[Target_Canvas][0]-up_img.width)//2, (canvas_map[Target_Canvas][1]-up_img.height)//2)
                    canvas.paste(up_img, offset)
                    canvas.save(res_img_path)

            status = "Success"
            error_msg = "None"
        except Exception as e:
            status = "Failed"
            error_msg = str(e)
            if Sort_Failures:
                (base_output / "flagged_failures").mkdir(exist_ok=True)
                shutil.copy(img_path, base_output / "flagged_failures" / img_path.name)

        # 🎯 Tracking individual file conversion (JSON details logged for Supabase)
        log_app_usage(
            app_name="upscaler_web",
            action="file_converted",
            details={
                "filename": img_path.name,
                "status": status,
                "error": error_msg,
                "time_taken": f"{round(time.time() - img_start, 2)}s"
            }
        )

        # Update CSV Report
        elapsed = round(time.time() - img_start, 2)
        with open(log_file, 'a', newline='') as f:
            csv.writer(f).writerow([img_path.name, Target_Canvas, f"{elapsed}s", status])

    print(f"\n✅ Process Complete! Total elapsed time: {round(time.time()-start_total, 2)}s")
    print(f"📍 Final output location: {base_output}")

    if not Use_Google_Drive:
        zip_name = f"/content/Gohard_Results_{timestamp}"
        shutil.make_archive(zip_name, 'zip', base_output)
        files.download(f"{zip_name}.zip")

start_upscale()

2026-04-10 05:45:58.663 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-10 05:45:58.667 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-10 05:45:58.668 Session state does not function when running a script without `streamlit run`
2026-04-10 05:45:58.703 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-10 05:45:58.704 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-10 05:45:58.704 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-10 05:45:58.728 
  command:

    streamlit run /usr/local/lib/python3.12/dist-packages/colab_kernel_launcher.py [ARGUMENTS]
2026-04-10 05:45:58.728 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-10

/content/Real-ESRGAN
Mounted at /content/drive
📂 Results for this session will be saved at: /content/drive/MyDrive/Gohard_Upscaler_Results/20260410_054558
👇 Select the images you want to restore (Batch upload supported)


Saving ligo1447546472.jpg to ligo1447546472.jpg
Saving ligo1447595130.jpg to ligo1447595130.jpg
Saving ligo1449291811.jpg to ligo1449291811.jpg
Downloading: "https://github.com/xinntao/Real-ESRGAN/releases/download/v0.1.0/RealESRGAN_x4plus.pth" to /content/Real-ESRGAN/weights/RealESRGAN_x4plus.pth

100% 63.9M/63.9M [00:00<00:00, 471MB/s]
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)
Downloading: "https://github.com/xinntao/facexlib/releases/download/v0.1.0/detection_Resnet50_Final.pth" to /content/Real-ESRGAN/gfpgan/weights/detecti

2026-04-10 05:47:04.746 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-10 05:47:04.747 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-10 05:47:04.749 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-10 05:47:04.750 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-10 05:47:04.753 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-10 05:47:04.754 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-10 05:47:04.754 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-10 05:47:04.755 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bar

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)
Testing 0 ligo1447546472
	Tile 1/2
	Tile 2/2


2026-04-10 05:47:16.575 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-10 05:47:16.577 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-10 05:47:16.578 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-10 05:47:16.580 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-10 05:47:16.581 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-10 05:47:16.582 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-10 05:47:16.583 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-10 05:47:16.585 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bar

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)
Testing 0 ligo1447595130
	Tile 1/1


2026-04-10 05:47:25.998 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-10 05:47:25.999 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-10 05:47:26.000 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-10 05:47:26.001 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-10 05:47:26.003 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-10 05:47:26.005 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-10 05:47:26.006 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-10 05:47:26.007 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bar


✅ Process Complete! Total elapsed time: 47.11s
📍 Final output location: /content/drive/MyDrive/Gohard_Upscaler_Results/20260410_054558
